# Build a Knowledge Graph for Your Spanning-Tree Network — the guided version

*A warm, step-by-step lab for switching engineers. You bring the STP. We bring the graph.*

You already run a graph algorithm on your switches every single day. It is called
**spanning tree**. STP builds a loop-free **tree** over your bridges: every switch is a
**node**, every link is a *possible* **edge**, and the root election plus the per-port
forwarding/blocking decision picks which edges actually carry traffic. You have been
operating a distributed graph algorithm your whole career; nobody just called it that.

A **knowledge graph** is that *same picture* — except **you** choose the nodes. And that
one freedom changes everything, because it lets you add the single node STP was never
designed to hold: the **business service** that dies when the wrong port goes to blocking.

By the end of this notebook you will have built, from an empty page, a small graph that
answers the question no `show spanning-tree` output can:

> *"acc-sw1 just lost its only forwarding uplink. Which business service is now at risk?"*

and watched it print **`Payroll App`** — a fact that lives in nobody's bridge table.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is
just **structured facts** (things, and the named links between them) plus **queries** that
walk those links. Everything is **deterministic and inspectable** — run it twice, get the
same answer, and you can point at every node that produced it. If you can read
`show spanning-tree`, you can read every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python +
networkx + matplotlib** (both already installed in Colab). No database, no Docker, no
credentials. Press *Runtime -> Run all*, or run each cell in order with **Shift+Enter**.


## The whole vocabulary — five words, each one an STP thing you already know

Before any code, here is the *entire* mental model. Everything after this is these five
ideas, repeated. And notice: you don't have to learn what any of them *mean* — you only
have to learn which STP thing each one maps onto.

| Graph word | Plain meaning | The STP thing it already is |
|---|---|---|
| **node** | a thing | a switch, a VLAN, a service |
| **edge** | a named, directed link between two nodes | a link between switches, "this switch connects that one" |
| **label** | a node's *type* | `Switch`, `VLAN`, `Service` — like "root bridge vs access switch" |
| **property** | a fact stored *on* a node or edge | `priority=4096`, `state='blocking'`, `vlan_id=10` |
| **traversal** | walking edges to answer a question | exactly what STP does — electing a root, then pruning loops |

That's it. Hold those five words and the rest is just your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick: **a fact can live on an
edge, not just a node.** "This port is blocking" is not really a property of one switch or
the other — it is a property of the *link between them*. STP knows this in its bones: port
state is per-port, per-link, decided for *that* segment. In a graph you write it down
literally: the port state lives **on the edge**. Keep an eye out for that moment — it is
where a topology diagram turns into something you can *query*.


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in
memory. We will use a **`DiGraph`** — a *directed* graph, where every edge has a
direction, an arrow. That matters to us: `acc-sw1 -> dist-sw1` ("acc-sw1 connects up to
dist-sw1") is a different statement from `dist-sw1 -> acc-sw1`. Spanning tree is full of
directional truth — who is the root, which way a root port points, which end blocks — so
directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab — nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print('Empty graph ready:', G)
print('Nodes:', G.number_of_nodes(), ' Edges:', G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty
`G` is your blank whiteboard. Every step from here just adds nodes and edges to it.


## Step 1 — Switches and their roles

**Theory.** A switch's *role* in the spanning tree is not the same as the switch. The same
box is the **root bridge** for one design and a plain non-root for another, decided by an
**election**: lowest bridge priority (then lowest MAC) wins, and everyone else measures
their best path *toward* that winner. You already think this way — "is core-sw1 the root
for VLAN 10?" is a question about a *role*, not a serial number.

So in the graph, `role` and `priority` are **properties** on the switch node. We add three:

- **core-sw1** — priority `4096`, the lowest, so it wins the election and is **root**.
- **dist-sw1** — a distribution switch, priority `32768` (the default), non-root.
- **acc-sw1** — the access switch where endpoints actually plug in, non-root.

Here is the design tension we're deliberately baking in: **acc-sw1 reaches the rest of the
network through essentially one active path.** In a real network its second uplink is a
hot standby that STP keeps *blocked* to avoid a loop. We model exactly that — so when the
active path fails, the blast radius shows up cleanly on a single edge you can point at.


In [ ]:
# add_node(name, **properties). 'role' and 'priority' are facts on each switch.
G.add_node('core-sw1', label='Switch', role='root',     priority=4096)   # lowest priority => root
G.add_node('dist-sw1', label='Switch', role='non-root', priority=32768)
G.add_node('acc-sw1',  label='Switch', role='non-root', priority=32768)  # access: where endpoints live

for n, d in G.nodes(data=True):
    if d.get('label') == 'Switch':
        print(f'{n}: role={d["role"]}, priority={d["priority"]}')

**Checkpoint.** Three switches print with their roles: `core-sw1` is `root` (priority
`4096`), the other two are `non-root`. They're floating unconnected for now — first we give
them something to be a tree *for*.


## Step 2 — The VLAN, and who is root for it

**Theory.** Spanning tree doesn't run in the abstract — it runs *per VLAN* (one tree per
VLAN, in the common PVST/RPVST world). So the VLAN is a first-class **node**. We fold the
STP *instance* into the VLAN node itself, because in a small design it's almost always 1:1
— one VLAN, one tree.

And the *result* of the root election — "core-sw1 is the root **for VLAN 10**" — is a fact
about the relationship between a switch and a VLAN. So it becomes an **edge**: a `ROOT_FOR`
link from `core-sw1` to `VLAN 10`. Read it literally: *"core-sw1 is root for VLAN 10."*

Notice we are *not* dumping the MAC address table or the BPDU stream in here. A node per
VLAN — not a node per learned MAC. We store the *shape* of the design, and we'll come back
to that principle by name later.


In [ ]:
G.add_node('VLAN 10', label='VLAN', vlan_id=10)
G.add_edge('core-sw1', 'VLAN 10', rel='ROOT_FOR')   # core-sw1 won the root election for VLAN 10

print('VLAN facts:', G.nodes['VLAN 10'])
print('Root edge:', [f'{u} -ROOT_FOR-> {v}'
      for u, v, d in G.edges(data=True) if d.get('rel') == 'ROOT_FOR'])

**Checkpoint.** The VLAN node shows `vlan_id=10`, and one `ROOT_FOR` edge says
`core-sw1 -ROOT_FOR-> VLAN 10`. You've written the root-election *outcome* into the graph
as a fact — no need to re-run the election, you just record who won. Next comes the links,
and the property the whole lab pivots on.


## Step 3 — Links: the edge carries the port state (this is the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. When STP puts a port into
**blocking**, *where* does that fact belong? Not on the switch (the switch is fine). Not on
the VLAN (the VLAN is fine). It belongs on the **link between two switches** — that's what
a port state describes. STP already agrees with you: forwarding vs blocking is decided
per-port, for *that* segment.

So we add **edges** with a `rel` of `CONNECTS`, and we hang a `state` **property** right on
each edge. Read `acc-sw1 --CONNECTS(blocking)--> core-sw1` literally: *"acc-sw1 connects to
core-sw1, and STP is holding that port in blocking."*

The wiring, mirroring the field design:

- `acc-sw1 -> dist-sw1` is **forwarding** — the active uplink carrying VLAN 10 today.
- `acc-sw1 -> core-sw1` is **blocking** — the redundant second uplink, STP-blocked so there's
  no loop. This is *normal*: a healthy standby, not a fault.
- `dist-sw1 -> core-sw1` is **forwarding** — distribution up to the root.

That blocking edge is worth pausing on. A blocked port is not an outage — it is STP doing
its job. Your switch logs it either way. In a moment you'll watch the graph tell the
difference between a *by-design* blocked port and one that *matters*.


In [ ]:
# add_edge(source, target, **properties). The port state is a property ON the edge.
G.add_edge('acc-sw1',  'dist-sw1', rel='CONNECTS', state='forwarding')  # the active uplink
G.add_edge('acc-sw1',  'core-sw1', rel='CONNECTS', state='blocking')    # redundant, STP-blocked (no loop)
G.add_edge('dist-sw1', 'core-sw1', rel='CONNECTS', state='forwarding')  # distribution up to the root

for u, v, d in G.edges(data=True):
    if d.get('rel') == 'CONNECTS':
        print(f'{u} --CONNECTS({d["state"]})--> {v}')

**Checkpoint.** Three `CONNECTS` edges print, and exactly one reads `blocking`: the
`acc-sw1 --CONNECTS(blocking)--> core-sw1` line. That single property on an edge is the
STP port state — the thing you'll toggle later to ask "what if?"


## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the
picture. We'll colour nodes by their **label** (switches one colour, the VLAN another) and
draw any **blocking** edge as a dashed red arrow — the same visual convention as a down
link, because a blocked port isn't carrying traffic right now either. This is the same
instinct as a topology diagram — except this diagram is generated *from the data*, so it
can never drift out of sync with the truth.


In [ ]:
def draw(G, title='STP knowledge graph'):
    colors = {'Switch': '#3aa0ff', 'VLAN': '#0f7f74', 'Service': '#c8702a'}
    node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
    pos = nx.spring_layout(G, seed=5)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(10, 7))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=9)

    blk   = [(u, v) for u, v, d in G.edges(data=True) if d.get('state') == 'blocking']
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in blk]
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16, edge_color='#8894a0')
    nx.draw_networkx_edges(G, pos, edgelist=blk,   arrows=True, arrowsize=16,
                           edge_color='#d34b3f', width=2.0, style='dashed')
    nx.draw_networkx_edge_labels(G, pos, font_size=7,
        edge_labels={(u, v): d.get('rel', '') for u, v, d in G.edges(data=True)})

    plt.axis('off'); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the switches (blue) and the VLAN (teal) with
`CONNECTS` and `ROOT_FOR` arrows, and one **dashed red** arrow from `acc-sw1` to `core-sw1`
— the standby, blocked uplink. This is still just a topology. It becomes a *knowledge*
graph in the next two steps, when we add the things a topology diagram has never carried:
where the VLAN reaches endpoints, and the business behind it.


## Step 4 — Where the VLAN reaches endpoints

**Theory.** A VLAN is not useful in the abstract; it matters at the **access switch where
endpoints actually plug in**. That's the switch whose survival the business cares about. On
a topology diagram this is implicit — you just *know* the phones hang off acc-sw1. In a
knowledge graph we make it explicit, because implicit knowledge is exactly what evaporates
at 3 AM.

So we add one **edge**: `VLAN 10 --REACHED_VIA--> acc-sw1`, meaning *"VLAN 10 is delivered
to its endpoints via acc-sw1."* This is the hinge the blast-radius query will swing on: if
acc-sw1 can't forward, then everything on VLAN 10 that lives behind it is stranded — no
matter how healthy the root and the core still are.


In [ ]:
G.add_edge('VLAN 10', 'acc-sw1', rel='REACHED_VIA')   # endpoints on VLAN 10 plug into acc-sw1

print('Edges out of VLAN 10:')
for u, v, d in G.edges('VLAN 10', data=True):
    print(f'  {u} --{d["rel"]}--> {v}')

**Checkpoint.** VLAN 10 now has an outgoing `REACHED_VIA` edge to `acc-sw1`. The graph now
knows *where* the VLAN meets the world. One node still missing — the reason any of this
matters to someone who signs the cheques.


## Step 5 — The business service: the node STP never had

**Theory.** This is the node your switches were never designed to hold, and the reason the
whole lab exists. STP knows bridges, ports, priorities, BPDUs, port states. It has never
once known that VLAN 10 is where the **Payroll App** lives, and that when acc-sw1 loses its
last forwarding path, **people don't get paid**.

So we add a **Service** node, `Payroll App`, with a `criticality` property, and we wire it
to the VLAN with a `RUNS_ON` edge: *"this service runs on that VLAN."* One edge — and now a
Layer 2 fact and a business fact are welded together in a place you can query. No
`show spanning-tree` output holds this link. It has never lived anywhere except tribal
knowledge. You're about to make it a first-class, walkable fact.


In [ ]:
G.add_node('Payroll App', label='Service', criticality='critical')
G.add_edge('Payroll App', 'VLAN 10', rel='RUNS_ON')

print('Graph now has', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges.')
print('The load-bearing link:', [f'{u} -RUNS_ON-> {v}'
      for u, v, d in G.edges(data=True) if d.get('rel') == 'RUNS_ON'])
draw(G, title='STP knowledge graph — now with the business service')

**Checkpoint.** The graph has grown an orange `Service` node, `Payroll App`, joined by a
`RUNS_ON` edge to the VLAN. In the redrawn picture you can *trace with your finger*:
Payroll App -> VLAN 10 -> acc-sw1 -> its uplinks. In the next step we make the computer
trace it for you.


## One honest caveat before we query it

> **This graph is a *snapshot* of last-known port states — not a live convergence engine.**

Real STP is a moving thing: pull a cable and within milliseconds to seconds RSTP
re-elects, unblocks the standby port, and reconverges around the failure. Our graph does
**not** simulate that. It holds the port states *as they were last recorded* and answers
"given these states, what is exposed?"

That is still enormously useful — it's exactly the question you ask in a design review or
the first minute of an incident ("if this is the state right now, who's affected?"). Just
keep the boundary honest: the graph reasons about **a snapshot**, and the value it adds is
the **business context** stitched onto that snapshot, not a prediction of how the tree will
reconverge. When you toggle a port later, you're asking "what does the world look like in
*this* state?" — not "what will STP do next?"


## Step 6 — The 3 AM question: blast radius as a traversal

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to
answer a question — the same kind of thing STP does when it walks the tree, except *you*
decide the walk. Our walk answers:

> *"For any access switch that has **no forwarding uplink left**, which business services on
> its VLAN are at risk?"*

The route the walk takes:

1. find a VLAN and the access switch it is `REACHED_VIA`;
2. check whether that access switch still has **any** `CONNECTS` edge in state `forwarding`;
3. if it does — nothing at risk, skip it (a blocked *standby* is fine);
4. if it does **not** — walk back to the VLAN and collect every service that `RUNS_ON` it.

Every hop is one edge. Crucially, step 2 is **conditioned on port state** — so a switch with
a healthy forwarding path returns nothing, and only a switch that has truly run out of
forwarding paths surfaces its services. Run it, and read the result carefully.


In [ ]:
def blast_radius(G):
    """Services put at risk by any access switch with zero forwarding paths left."""
    at_risk = []
    for vlan, acc, d in G.edges(data=True):
        # 1) a VLAN delivered to endpoints via some access switch
        if d.get('rel') != 'REACHED_VIA':
            continue
        # 2) does that access switch still have ANY forwarding uplink?
        has_forwarding = any(
            e.get('rel') == 'CONNECTS' and e.get('state') == 'forwarding'
            for _, _, e in G.out_edges(acc, data=True)
        )
        if has_forwarding:
            continue
        # 3) no forwarding path left -> which services ride this VLAN?
        for svc, _, d3 in G.in_edges(vlan, data=True):
            if d3.get('rel') == 'RUNS_ON':
                at_risk.append((acc, vlan, svc))
    return at_risk

hits = blast_radius(G)
for acc, vlan, svc in hits:
    print(f'{acc} has NO forwarding path  ->  {vlan}  ->  AT RISK: {svc}')
if not hits:
    print('nothing at risk — every access switch still has a forwarding path')

**Read the result — this is the payoff.** It printed **nothing at risk**, and that is
*exactly right*. acc-sw1 has a blocked standby port to core-sw1, and a switch will happily
log and alarm on that blocked port — but it is **not** an outage. Your graph knew the
difference: acc-sw1 still forwards through dist-sw1, so Payroll App is safe.

That is already worth the whole exercise: the graph separates *by-design blocking* (noise)
from *actual exposure* (signal). A `show spanning-tree` gives you the port states; only the
graph, with the service stitched on, tells you which blocked ports you can ignore and which
would end your night. Now let's make one *matter*.


## Step 7 — What-if: break the last forwarding path, then restore it

**Theory.** Because the port state lives on an edge as a plain property, you can *change* it
and re-ask — running "what if this fails?" (or "what if I fix it?") on a **model**, safely,
with no one paged and no maintenance window. Flip acc-sw1's active uplink to `blocking`:
now *both* of its uplinks are blocked (the standby was already blocked), it has no
forwarding path, and Payroll App surfaces. Flip it back to `forwarding`: the risk clears.

We'll toggle it a few times and, at the end, deliberately **leave it broken** — so the rest
of the lab builds on a live incident and you can watch the blast radius *grow* as we add
more truth.


In [ ]:
# You can read and write an edge's properties directly by [source][target].
G['acc-sw1']['dist-sw1']['state'] = 'blocking'    # the only forwarding uplink just failed
print('after failure:', [svc for *_, svc in blast_radius(G)] or 'nothing at risk')

G['acc-sw1']['dist-sw1']['state'] = 'forwarding'  # STP reconverges / the fiber is repaired
print('after repair: ', [svc for *_, svc in blast_radius(G)] or 'nothing at risk')

G['acc-sw1']['dist-sw1']['state'] = 'blocking'    # break it once more — left here so the next steps build on it
print('re-broken:    ', [svc for *_, svc in blast_radius(G)])

**Checkpoint.** After the failure you should see `['Payroll App']`; after the repair,
`nothing at risk`; after re-breaking it, `['Payroll App']` returns. Same graph, same query
— only the `state` on one edge changed, and the answer *responded*. That is what makes it a
model rather than a drawing.


## Your turn #1 — a second service on the same VLAN

Real VLANs rarely carry just one thing. Suppose an **HR Portal** service also rides on
VLAN 10. Add it, wire it with a `RUNS_ON` edge, and re-run `blast_radius`. Because acc-sw1
is still broken from Step 7, you should now see **two** services surface from the same
failure — one edge of extra truth is all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want
to check.


In [ ]:
# TODO: add an 'HR Portal' Service node (criticality='high'),
#       then a RUNS_ON edge from 'HR Portal' to 'VLAN 10'.

# G.add_node(...)
# G.add_edge(...)

for acc, vlan, svc in blast_radius(G):
    print(f'AT RISK: {svc}  (on {vlan}, via {acc})')

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('HR Portal', label='Service', criticality='high')
G.add_edge('HR Portal', 'VLAN 10', rel='RUNS_ON')

for acc, vlan, svc in blast_radius(G):
    print(f'AT RISK: {svc}  (on {vlan}, via {acc})')
```

Now both `Payroll App` **and** `HR Portal` come back from the one broken access switch. The
blast radius grew the instant you told the graph one more true thing — you didn't touch the
query at all.
</details>


In [ ]:
# (Solution, applied — so the rest of the notebook has both services in the graph.)
G.add_node('HR Portal', label='Service', criticality='high')
G.add_edge('HR Portal', 'VLAN 10', rel='RUNS_ON')
print('Services at risk now:', sorted({svc for *_, svc in blast_radius(G)}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"A `Switch` has a `priority`. A `VLAN` is `REACHED_VIA` a `Switch`. A `Service`
  `RUNS_ON` a `VLAN`. A link between switches carries a `state`."* — these are the **rules**:
  which node types exist, which edges are allowed, what shape a valid fact takes. That is
  the **schema**. Its fancy name is an **ontology** — and here's the friendly definition:
  *an ontology is the rulebook of your graph, the design written down as structure.* You
  already trust IEEE 802.1D/802.1w to say what a valid spanning tree is; an ontology does
  the same job for your graph.

- *"This particular switch is `acc-sw1`, priority `32768`, and its uplink to dist-sw1 is
  `blocking`."* — these are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a hostname, a VLAN ID, or
a MAC, it is data (an instance), never schema.** `Switch` is schema; `acc-sw1` is data.
`RUNS_ON` is schema; "Payroll App runs on VLAN 10" is data. Keep the vocabulary small and
stable; let the instances be many. That single discipline is the difference between a graph
you can grow for years and a swamp you abandon in a month.


## A peek at the real thing (1/2) — the same switches in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built
are exactly what you'd build in a real graph database like **Neo4j**, whose query language
is **Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been
drawing — `(node)-[:EDGE]->(node)`.

Here is **Step 3 (the links)** as Cypher. `MERGE` means "find this node or create it if
missing" (so re-running is safe); `SET` assigns properties. This is *reference only* — you
don't run it here, it just shows you the same idea in the grown-up tool:

```cypher
MERGE (core:Switch {id: 'core-sw1'})  SET core.role = 'root',     core.priority = 4096;
MERGE (dist:Switch {id: 'dist-sw1'})  SET dist.role = 'non-root', dist.priority = 32768;
MERGE (acc:Switch  {id: 'acc-sw1'})   SET acc.role  = 'non-root', acc.priority  = 32768;

// the port state, as a property on the relationship — same as our edge.
// MERGE the edge, then SET the state (mirroring the node idiom) so re-running
// updates the property in place instead of creating a duplicate edge.
MERGE (acc)-[r1:CONNECTS]->(dist)  SET r1.state = 'forwarding';
MERGE (acc)-[r2:CONNECTS]->(core)  SET r2.state = 'blocking';
```

See it? `(acc)-[:CONNECTS {state:'blocking'}]->(core)` is character-for-character the same
statement as our `G.add_edge('acc-sw1', 'core-sw1', rel='CONNECTS', state='blocking')`. Same
node, same edge, same fact-on-the-edge — one just happens to live in a database that scales
to millions of nodes.


## A peek at the real thing (2/2) — the 3 AM question in Cypher

Our `blast_radius` walk is a ~15-line Python function. In Cypher, that same traversal is a
few lines, because in a graph database you **draw the shape you're looking for** and let the
engine find every match — no manual loops:

```cypher
MATCH (vlan:VLAN)-[:REACHED_VIA]->(acc:Switch)
WHERE NOT (acc)-[:CONNECTS {state: 'forwarding'}]->()
MATCH (svc:BusinessService)-[:DEPENDS_ON]->(vlan)
RETURN acc.name AS access_switch, vlan.vlan_id AS vlan,
       collect(DISTINCT svc.name) AS services_at_risk;
```

Read the `WHERE` line as a sentence: *"...where the access switch has no forwarding
CONNECTS edge left."* That's the same step 2 you wrote in Python — the pattern you `MATCH`
**is** the traversal.

One honest note on vocabulary: this project's real STP ontology names that last hop
`BusinessService -[:DEPENDS_ON]-> VLAN`, where our tiny graph used `RUNS_ON`. Same fact,
named by each system's own rulebook — which is exactly the schema-vs-instance point from
the reveal above. Different engine, different word, identical thinking. If you can read the
Python, you can read the Cypher — you already speak this language.


## Your turn #2 — make the query respond to a *different* failure

A second access switch, `acc-sw2`, carries **VLAN 20** with its own service, a **Time
Clock**. Right now it's perfectly healthy — its uplink to dist-sw1 is `forwarding`, so
nothing on it is at risk. Let's prove the model reacts to reality:

1. add `VLAN 20`, `acc-sw2`, and a `Time Clock` service, wired the same way acc-sw1 was
   (a forwarding uplink to dist-sw1, a blocking standby to core-sw1, `REACHED_VIA`, `RUNS_ON`);
2. re-run `blast_radius` — Time Clock should **not** appear (acc-sw2 still forwards);
3. now set `acc-sw2 -> dist-sw1` to `blocking` and re-run — Time Clock **appears**.

This is the whole point of Step 7, in your own hands: the answer follows the state. Fill in
the `# TODO`s, then run.


In [ ]:
# TODO 1: add VLAN 'VLAN 20' (vlan_id=20), access switch 'acc-sw2' (role='non-root',
#         priority=32768), and Service 'Time Clock' (criticality='high'). Then wire:
#   - CONNECTS(forwarding): 'acc-sw2' -> 'dist-sw1'   (a healthy uplink)
#   - CONNECTS(blocking):   'acc-sw2' -> 'core-sw1'   (standby)
#   - REACHED_VIA:          'VLAN 20' -> 'acc-sw2'
#   - RUNS_ON:              'Time Clock' -> 'VLAN 20'

# G.add_node(...); G.add_node(...); G.add_node(...)
# G.add_edge(...); ...

print('Currently at risk:', sorted({s for *_, s in blast_radius(G)}))

# TODO 2: break acc-sw2's forwarding uplink (G['acc-sw2']['dist-sw1']['state'] = 'blocking')
#         and print blast_radius again — Time Clock should join the list.

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('VLAN 20', label='VLAN', vlan_id=20)
G.add_node('acc-sw2', label='Switch', role='non-root', priority=32768)
G.add_node('Time Clock', label='Service', criticality='high')
G.add_edge('acc-sw2', 'dist-sw1', rel='CONNECTS', state='forwarding')
G.add_edge('acc-sw2', 'core-sw1', rel='CONNECTS', state='blocking')
G.add_edge('VLAN 20', 'acc-sw2', rel='REACHED_VIA')
G.add_edge('Time Clock', 'VLAN 20', rel='RUNS_ON')

print('With acc-sw2 healthy:  ', sorted({s for *_, s in blast_radius(G)}))
# -> HR Portal, Payroll App  (Time Clock is safe: acc-sw2 still forwards)

G['acc-sw2']['dist-sw1']['state'] = 'blocking'
print('With acc-sw2 uplink down:', sorted({s for *_, s in blast_radius(G)}))
# -> now Time Clock joins the list
```

Before you broke the link, Time Clock was *in the graph* but *not at risk* — the query
correctly ignored a healthy path. The moment acc-sw2 lost its last forwarding uplink, the
exact same query surfaced it. You didn't teach the query about VLAN 20; you just told the
graph the truth, and the traversal did the rest.
</details>


In [ ]:
# (Solution, applied — breaks acc-sw2 to show Time Clock, then restores it to a clean state.)
G.add_node('VLAN 20', label='VLAN', vlan_id=20)
G.add_node('acc-sw2', label='Switch', role='non-root', priority=32768)
G.add_node('Time Clock', label='Service', criticality='high')
G.add_edge('acc-sw2', 'dist-sw1', rel='CONNECTS', state='forwarding')
G.add_edge('acc-sw2', 'core-sw1', rel='CONNECTS', state='blocking')
G.add_edge('VLAN 20', 'acc-sw2', rel='REACHED_VIA')
G.add_edge('Time Clock', 'VLAN 20', rel='RUNS_ON')

print('acc-sw2 healthy:     ', sorted({s for *_, s in blast_radius(G)}))
G['acc-sw2']['dist-sw1']['state'] = 'blocking'
print('acc-sw2 uplink down: ', sorted({s for *_, s in blast_radius(G)}))
G['acc-sw2']['dist-sw1']['state'] = 'forwarding'   # restore acc-sw2 health
print('acc-sw2 restored:    ', sorted({s for *_, s in blast_radius(G)}))

## Make it yours — swap in *your* network

Now the moment it becomes your tool, not mine. Change the four values below to **one** VLAN,
**one** access switch, and **one** service *you* actually run. We add the failed uplink for
you (a single blocked port and no other path), so your service shows up. Run it, and watch
**your own service name** come back from `blast_radius` — proof that the machine you built
understands your network, not just the demo's.

Keep it to the smallest slice that answers one real question. You can always add one more
node tomorrow — that's the whole promise of a graph you can grow.


In [ ]:
# --- change these four values to your network ---
MY_VLAN    = 'VLAN 42'
MY_ACCESS  = 'campus-acc-1'
MY_ID      = 42
MY_SERVICE = 'Badge Access'
# ------------------------------------------------

G.add_node(MY_VLAN,    label='VLAN',    vlan_id=MY_ID)
G.add_node(MY_ACCESS,  label='Switch',  role='non-root', priority=32768)
G.add_node(MY_SERVICE, label='Service', criticality='critical')

G.add_edge(MY_ACCESS, 'core-sw1', rel='CONNECTS', state='blocking')  # the failure you're modelling: only uplink, blocked
G.add_edge(MY_VLAN,   MY_ACCESS,  rel='REACHED_VIA')
G.add_edge(MY_SERVICE, MY_VLAN,   rel='RUNS_ON')

print(f'If {MY_ACCESS} loses its last forwarding uplink, these services are at risk:')
for acc, vlan, svc in blast_radius(G):
    if acc == MY_ACCESS:
        print(f'  AT RISK: {svc}   (on {vlan})')

**Checkpoint.** Your own service name — `Badge Access`, or whatever you typed — prints as
*at risk*. That is the moment the tool stopped being a tutorial and became yours. Every
other service you run is just three more lines away.


## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your design review, not the numbers of your monitoring dashboard.**

Switches, roles, VLANs, port states, services, gateways, guards — the **nouns** you'd draw
on a whiteboard while arguing about a design — those belong in the graph. Per-port packet
counters, CPU percentages, the full MAC address table, the BPDU-per-second stream — those
are the **numbers**; leave them in the systems that already store them well, and let the
graph *reference* them when it needs to. The graph holds the *shape of the dependency*;
your NMS holds the firehose. Keep that line sharp and your graph stays queryable for years.

### Where to go next
- **The companion Neo4j lab** does this exact thing in a real graph database — same nodes,
  same edges, same 3 AM question — so the two Cypher peeks above become things you run.
- **Add a protocol.** The shapes generalize: an FHRP group is a node the VLAN points at for
  its gateway; a port-channel is an edge bundling members; a trunk is an edge that carries
  VLANs. The same five words carry all of it.
- **Add the change layer.** Model a `ChangeEvent` node linked to what it touched, and "what
  changed right before this VLAN went dark?" becomes one more traversal.


## You built a knowledge graph

Look back at the distance. You started with an empty page and five plain words. You added
switches, a VLAN, and their links — a topology. Then you added where the VLAN reaches
endpoints and the one node STP never had, the **business service** — and the topology
became *knowledge*. Finally you asked it the question no `show spanning-tree` can answer,
and it correctly said *nothing at risk* while a port was merely a by-design standby, then
surfaced **Payroll App** the instant the last forwarding path failed — and responded
correctly every time you changed the world underneath it.

The important idea was never STP, and never networkx. It's this: **operational truth has a
shape** — a service runs on a VLAN, a VLAN is reached via an access switch, a switch
connects to another with a port state — and once that shape is a graph, impact analysis
stops being tribal knowledge and becomes a traversal.

You already run a graph algorithm on your switches every day. Now you know how to build the
one that holds the part spanning tree was never asked to remember. Go model one real
service on your network, and ask it what breaks.
